# DSPIN setup — pilot selection & AnnData prep

**Design:** `~/.gstack/projects/wildrxspf-brains/jboktor-master-design-20260809-165240.md`

Pipeline:
1. R `export_dspin_h5ads.R --mode candidates` → `candidate_cellcounts.csv`
2. R `rank_dspin_pilots.R` → `pilot_candidates_draft.csv` + `pilot_manifest.csv` (≤5 interactive)
3. R `export_dspin_h5ads.R --mode export-pilot` → `raw_counts.h5ad` per slice
4. **This notebook:** verify obs columns, normalize + HVG top 2000, write `filtered.h5ad`

Locked wiring: `sample_id=sample`, `if_control=(microbiome==SPF)`, `batch=tissue`. Unit: `(supertype_name, tissue)`.


In [ ]:
from pathlib import Path
import json
import warnings

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2

WKDIR = Path("/resnick/groups/mthomson/jboktor/WILDRxSPF_brains")
DSPIN_DIR = WKDIR / "data/interim/dspin"
DSPIN_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = DSPIN_DIR / "pilot_manifest.csv"
DRAFT = DSPIN_DIR / "pilot_candidates_draft.csv"
CAND = DSPIN_DIR / "candidate_cellcounts.csv"

MIN_CELLS_GENE = 10
N_HVG = 2000

print("DSPIN_DIR:", DSPIN_DIR)
print("manifest exists:", MANIFEST.exists())
print("draft exists:", DRAFT.exists())
print("candidates exists:", CAND.exists())


## 0. Env check

`dspin` is used in the models notebook; here we only need `anndata` + `scanpy`.


In [ ]:
import sys
print(sys.executable)
print("anndata", ad.__version__, "scanpy", sc.__version__)
try:
    from dspin.dspin import DSPIN  # noqa: F401
    print("dspin: import OK")
except Exception as e:
    print("dspin: not importable here (OK for setup) —", type(e).__name__, e)


## 1. Load / preview ranking tables

If missing, run on a high-mem node:

```bash
sbatch notebooks/shell_scripts/run_dspin_export.sh all
# or step-wise: candidates → rank → export-pilot
```


In [ ]:
if CAND.exists():
    cand = pd.read_csv(CAND)
    print("candidates:", cand.shape)
    print(cand.head())
    print("hard_pass:", int(cand["hard_pass"].sum()) if "hard_pass" in cand else "n/a")
else:
    cand = None
    print("MISSING", CAND)

if DRAFT.exists():
    draft = pd.read_csv(DRAFT)
    print("draft:", draft.shape)
    print(draft)
else:
    draft = None
    print("MISSING", DRAFT, "— run rank_dspin_pilots.R")

if MANIFEST.exists():
    manifest = pd.read_csv(MANIFEST)
    print("manifest:", manifest.shape)
    print(manifest)
else:
    manifest = None
    print("MISSING", MANIFEST)


## 2. Prepare each interactive slice (normalize + HVG)

Reads `raw_counts.h5ad` (cells × genes, raw UMI), writes `filtered.h5ad` with log1p + top 2000 HVGs. Updates `n_genes` in the manifest.


In [ ]:
def prepare_slice(h5ad_raw: Path, h5ad_out: Path) -> dict:
    adata = ad.read_h5ad(h5ad_raw)
    for col in ["sample_id", "batch", "if_control"]:
        if col not in adata.obs:
            raise KeyError(f"{h5ad_raw}: missing obs[{col}]")

    sc.pp.filter_genes(adata, min_cells=MIN_CELLS_GENE)
    adata.layers["counts"] = adata.X.copy()

    # HVG on counts (seurat_v3); fall back to seurat on log1p
    try:
        sc.pp.highly_variable_genes(
            adata, n_top_genes=N_HVG, flavor="seurat_v3", layer="counts", subset=False
        )
    except Exception as e:
        print("seurat_v3 HVG failed, falling back:", e)
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor="seurat", subset=False)
    else:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

    adata = adata[:, adata.var["highly_variable"]].copy()
    h5ad_out.parent.mkdir(parents=True, exist_ok=True)
    adata.write_h5ad(h5ad_out, compression="gzip")
    return {
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_SPF": int((adata.obs["microbiome"].astype(str) == "SPF").sum()),
        "n_WildR": int((adata.obs["microbiome"].astype(str) == "WildR").sum()),
        "n_sample_id": int(adata.obs["sample_id"].nunique()),
        "filtered": str(h5ad_out),
    }

if manifest is None:
    print("Skip prepare — no manifest")
else:
    rows = []
    for _, row in manifest.iterrows():
        slice_path = Path(row["slice_path"]) if "slice_path" in row and pd.notna(row["slice_path"]) else DSPIN_DIR / f"{row['tissue']}__{row['supertype_slug']}"
        raw = slice_path / "raw_counts.h5ad"
        out = slice_path / "filtered.h5ad"
        if not raw.exists():
            print("SKIP (no raw h5ad):", raw)
            rows.append({**row.to_dict(), "prep_status": "missing_raw"})
            continue
        try:
            info = prepare_slice(raw, out)
            print("OK", slice_path.name, info)
            rows.append({**row.to_dict(), **info, "prep_status": "ok"})
        except Exception as e:
            print("FAIL", slice_path.name, e)
            rows.append({**row.to_dict(), "prep_status": f"error:{type(e).__name__}"})
    manifest2 = pd.DataFrame(rows)
    manifest2.to_csv(MANIFEST, index=False)
    print("Updated", MANIFEST)
    cols = [c for c in ["rank", "tissue", "supertype_name", "n_genes", "prep_status", "filtered"] if c in manifest2.columns]
    print(manifest2[cols])


## 3. Sanity checks for models notebook

Confirm each filtered AnnData has SPF + WildR `sample_id`s with enough cells.


In [ ]:
def sample_floor_table(adata: ad.AnnData) -> pd.DataFrame:
    g = (
        adata.obs.groupby(["sample_id", "microbiome"], observed=True)
        .size()
        .reset_index(name="n_cells")
        .sort_values("n_cells")
    )
    return g

if manifest is not None and MANIFEST.exists():
    man = pd.read_csv(MANIFEST)
    for _, row in man.iterrows():
        if row.get("prep_status") != "ok":
            continue
        path = Path(row["filtered"]) if "filtered" in row and pd.notna(row.get("filtered")) else None
        if path is None or not path.exists():
            continue
        a = ad.read_h5ad(path)
        print("\n===", row["tissue"], row["supertype_name"], "===")
        print(sample_floor_table(a))
        print("batch unique:", a.obs["batch"].unique().tolist())
else:
    print("Nothing to check yet")
